# Tham số, siêu tham số và tinh chỉnh

Dữ liệu mô phỏng: $y = \sin(2\pi x) + \text{nhiễu}$. Mô hình: **hồi quy đa thức** bậc $p$, $\hat{y} = w_0 + w_1 x + w_2 x^2 + \dots + w_p x^p$.

- **Tham số**: $w_0, w_1, \dots, w_p$, được học từ dữ liệu bằng cách cực tiểu hàm mất mát.
- **Siêu tham số**: bậc đa thức $p$, do người dùng chọn trước khi huấn luyện.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression

def make_data(n, rng):
    """Sinh n điểm (x, y) với y = sin(2πx) + nhiễu, x trong [0, 1)."""
    x = np.sort(rng.uniform(0, 1, n))
    y = np.sin(2 * np.pi * x) + rng.normal(0, 0.2, n)
    return x.reshape(-1, 1), y

rng = np.random.default_rng(1)               # cố định bộ sinh số ngẫu nhiên
X_train, y_train = make_data(15, rng)        # tập huấn luyện: 15 điểm
X_val, y_val = make_data(15, rng)            # tập xác thực: 15 điểm
X_test, y_test = make_data(500, rng)         # tập kiểm tra: 500 điểm


def poly_model(degree):
    """Hồi quy đa thức bậc degree: chuẩn hóa x, tạo [x, x², …, x^degree], chuẩn hóa lại, rồi hồi quy tuyến tính."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), LinearRegression())


def rmse(model, X, y):
    """Căn bậc hai của sai số bình phương trung bình."""
    return np.sqrt(np.mean((model.predict(X) - y) ** 2))

## 1. Tham số được học từ dữ liệu

Cố định siêu tham số $p = 3$ rồi huấn luyện: `fit` tìm các tham số $w_j$ làm cực tiểu MSE trên tập huấn luyện.

In [2]:
model = poly_model(3).fit(X_train, y_train)
regression = model[-1]                       # bước hồi quy tuyến tính cuối pipeline
print('w0 =', round(regression.intercept_, 3))
print('w1, w2, w3 =', regression.coef_.round(3))   # hệ số trên đặc trưng đã chuẩn hóa
print('RMSE huấn luyện = %.3f' % rmse(model, X_train, y_train))

w0 = -0.005
w1, w2, w3 = [-1.578  0.174  1.143]
RMSE huấn luyện = 0.194


## 2. Tinh chỉnh siêu tham số bằng tập xác thực

Thử các bậc $p = 1, \dots, 12$. Với mỗi $p$: học tham số trên **tập huấn luyện**, đo sai số trên **tập xác thực**. Chọn $p$ có sai số xác thực nhỏ nhất. **Tập kiểm tra không được dùng** trong bước này.

In [3]:
degrees = range(1, 13)
train_errors, val_errors = [], []
for degree in degrees:
    model = poly_model(degree).fit(X_train, y_train)
    train_errors.append(rmse(model, X_train, y_train))
    val_errors.append(rmse(model, X_val, y_val))
    print(f'bậc {degree:2d}: RMSE huấn luyện = {train_errors[-1]:.3f}, RMSE xác thực = {val_errors[-1]:8.3f}')

best_degree = degrees[int(np.argmin(val_errors))]
print('Bậc tốt nhất theo tập xác thực:', best_degree)

bậc  1: RMSE huấn luyện = 0.495, RMSE xác thực =    0.427
bậc  2: RMSE huấn luyện = 0.495, RMSE xác thực =    0.427
bậc  3: RMSE huấn luyện = 0.194, RMSE xác thực =    0.173
bậc  4: RMSE huấn luyện = 0.187, RMSE xác thực =    0.174
bậc  5: RMSE huấn luyện = 0.178, RMSE xác thực =    0.169
bậc  6: RMSE huấn luyện = 0.163, RMSE xác thực =    0.264
bậc  7: RMSE huấn luyện = 0.163, RMSE xác thực =    0.267
bậc  8: RMSE huấn luyện = 0.136, RMSE xác thực =    1.088
bậc  9: RMSE huấn luyện = 0.135, RMSE xác thực =    1.525
bậc 10: RMSE huấn luyện = 0.110, RMSE xác thực =    4.652
bậc 11: RMSE huấn luyện = 0.104, RMSE xác thực =   12.351
bậc 12: RMSE huấn luyện = 0.074, RMSE xác thực =  150.575
Bậc tốt nhất theo tập xác thực: 5


Sau khi chọn xong siêu tham số, huấn luyện lại và đánh giá **một lần duy nhất** trên tập kiểm tra:

In [4]:
final_model = poly_model(best_degree).fit(X_train, y_train)
print('RMSE kiểm tra với bậc %d = %.3f' % (best_degree, rmse(final_model, X_test, y_test)))

RMSE kiểm tra với bậc 5 = 0.231


## 3. Kiểm định chéo k-fold và `GridSearchCV`

Với ít dữ liệu, một tập xác thực 15 điểm cho kết quả dao động. **Kiểm định chéo k-fold** gộp huấn luyện và xác thực, chia thành $k$ phần; lần lượt dùng $k - 1$ phần để huấn luyện, 1 phần để đánh giá, rồi lấy trung bình. `GridSearchCV` tự động thử mọi giá trị siêu tham số theo cách này.

In [5]:
from sklearn.model_selection import GridSearchCV, KFold

X_dev = np.vstack([X_train, X_val])          # gộp huấn luyện + xác thực: 30 điểm
y_dev = np.concatenate([y_train, y_val])

search = GridSearchCV(poly_model(1),
                      param_grid={'polynomialfeatures__degree': list(degrees)},
                      cv=KFold(n_splits=5, shuffle=True, random_state=0),
                      scoring='neg_root_mean_squared_error')
search.fit(X_dev, y_dev)
cv_rmse = -search.cv_results_['mean_test_score']   # RMSE trung bình qua 5 lần
for degree, error in zip(degrees, cv_rmse):
    print(f'bậc {degree:2d}: RMSE kiểm định chéo = {error:.3f}')
print('Bậc tốt nhất:', search.best_params_['polynomialfeatures__degree'])
print('RMSE kiểm tra = %.3f' % rmse(search.best_estimator_, X_test, y_test))

bậc  1: RMSE kiểm định chéo = 0.481
bậc  2: RMSE kiểm định chéo = 0.528
bậc  3: RMSE kiểm định chéo = 0.199
bậc  4: RMSE kiểm định chéo = 0.192
bậc  5: RMSE kiểm định chéo = 0.170
bậc  6: RMSE kiểm định chéo = 0.172
bậc  7: RMSE kiểm định chéo = 0.176
bậc  8: RMSE kiểm định chéo = 0.208
bậc  9: RMSE kiểm định chéo = 0.283
bậc 10: RMSE kiểm định chéo = 0.304
bậc 11: RMSE kiểm định chéo = 0.376
bậc 12: RMSE kiểm định chéo = 1.425
Bậc tốt nhất: 5
RMSE kiểm tra = 0.217
